In [1]:
import os
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import joblib

print("[OK] All core data science and machine learning packages loaded successfully.")

[OK] All core data science and machine learning packages loaded successfully.


In [2]:
# Resolve dataset path dynamically
possible_paths = [
    os.path.join("..", "data", "comprehensive_plants_and_crops.csv"),
    os.path.join("data", "comprehensive_plants_and_crops.csv"),
    "comprehensive_plants_and_crops.csv"
]
DATA_PATH = next((p for p in possible_paths if os.path.exists(p)), None)

if DATA_PATH is None:
    raise FileNotFoundError("Could not locate comprehensive_plants_and_crops.csv")

df = pd.read_csv(DATA_PATH)
print(f"Dataset Dimensions : {df.shape[0]} samples × {df.shape[1]} features")
print(f"Total Crop Classes : {df['label'].nunique()} distinct plants/crops")
print(f"Botanical Categories: {df['category'].unique().tolist()}")

df.head(8)

Dataset Dimensions : 4060 samples × 13 features
Total Crop Classes : 58 distinct plants/crops
Botanical Categories: ['Cereal', 'Pulse', 'Fruit', 'Vegetable', 'Cash Crop', 'Plantation', 'Oilseed', 'Spice']


,N,P,K,temperature,humidity,ph,rainfall,soil_type,category,growth_duration_days,water_requirement,sunlight_hours,label
0,83,47,42,27.81,81.06,6.31,277.38,Alluvial,Cereal,124,Very High,6.7,Rice
1,84,45,38,24.60,74.35,5.71,213.13,Clayey,Cereal,113,Very High,7.2,Rice
2,72,39,45,23.44,82.27,5.83,213.67,Clayey,Cereal,120,Very High,6.3,Rice
3,83,44,38,22.50,89.41,6.39,198.27,Alluvial,Cereal,124,Very High,6.3,Rice
4,81,36,34,24.49,84.95,6.47,226.53,Loamy,Cereal,118,Very High,6.1,Rice
5,74,45,44,24.86,74.95,6.53,218.45,Clayey,Cereal,115,Very High,7.4,Rice
6,88,53,36,23.23,83.33,6.79,215.62,Clayey,Cereal,118,Very High,6.3,Rice
7,70,52,45,23.82,86.01,6.54,210.65,Clayey,Cereal,122,Very High,7.9,Rice


In [3]:
# Statistical summary of numeric agronomic parameters
print("=== Descriptive Statistics ===")
display(df.describe().round(2))

# Samples per botanical category
cat_dist = df['category'].value_counts()
print("\n=== Samples per Category ===")
display(cat_dist)

=== Descriptive Statistics ===


,N,P,K,temperature,humidity,ph,rainfall,growth_duration_days,sunlight_hours
count,4060.00,4060.00,4060.0,4060.00,4060.00,4060.00,4060.00,4060.00,4060.00
mean,57.75,44.96,46.9,24.04,67.55,6.44,96.06,185.71,7.45
std,29.05,22.90,34.5,4.82,17.02,0.69,55.15,121.09,1.15
min,6.00,5.00,5.0,9.69,12.00,4.00,16.82,37.00,3.00
25%,33.00,30.00,26.0,20.44,57.29,6.01,57.00,95.00,6.70
50%,57.00,42.00,40.0,24.45,67.72,6.49,76.12,119.00,7.50
75%,79.00,54.00,56.0,27.39,80.92,6.90,117.16,338.00,8.30
max,164.00,150.00,214.0,40.53,100.00,8.63,291.82,511.00,10.40



=== Samples per Category ===


category
Fruit         910
Vegetable     840
Pulse         630
Cereal        560
Cash Crop     350
Spice         350
Oilseed       280
Plantation    140
Name: count, dtype: int64

In [4]:
# Separate raw features and target
X_raw = df.drop(columns=['label'])
y_raw = df['label']

# Label encode target
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_raw)
class_names = list(label_encoder.classes_)
print(f"Target classes encoded: {len(class_names)} classes.")

# One-hot encoding for categorical variables
categorical_cols = ['soil_type', 'category', 'water_requirement']
numerical_cols = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall', 'growth_duration_days', 'sunlight_hours']

X_encoded = pd.get_dummies(X_raw, columns=categorical_cols, drop_first=False)
feature_names = list(X_encoded.columns)
print(f"Total features after encoding: {len(feature_names)}")

# Stratified Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape[0]} records")
print(f"Testing set : {X_test.shape[0]} records")

# Standard scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Target classes encoded: 58 classes.
Total features after encoding: 28
Training set: 3248 records
Testing set : 812 records


In [5]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=30, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42, max_depth=20),
    "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5, weights='distance'),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

benchmark_records = []

for name, clf in models.items():
    print(f"Training {name}...")
    clf.fit(X_train_scaled, y_train)
    
    train_pred = clf.predict(X_train_scaled)
    test_pred = clf.predict(X_test_scaled)
    
    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)
    prec = precision_score(y_test, test_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, test_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, test_pred, average='weighted', zero_division=0)
    
    benchmark_records.append({
        "Model": name,
        "Train Accuracy (%)": round(train_acc * 100, 2),
        "Test Accuracy (%)": round(test_acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1 Score (%)": round(f1 * 100, 2)
    })

results_df = pd.DataFrame(benchmark_records)
results_df.sort_values(by="Test Accuracy (%)", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

print("\n=== Model Performance Comparison ===")
display(results_df)

Training Random Forest...
Training Gradient Boosting...
Training Decision Tree...
Training K-Nearest Neighbors...
Training Logistic Regression...

=== Model Performance Comparison ===


,Model,Train Accuracy (%),Test Accuracy (%),Precision (%),Recall (%),F1 Score (%)
0,Random Forest,100.00,99.26,99.37,99.26,99.26
1,Logistic Regression,98.12,97.78,97.96,97.78,97.76
2,Gradient Boosting,99.97,96.55,96.89,96.55,96.58
3,Decision Tree,99.97,95.94,96.17,95.94,95.91
4,K-Nearest Neighbors,100.00,95.32,95.79,95.32,95.19


In [6]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"Champion Model: {best_model_name}")

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(best_model, X_train_scaled, y_train, cv=cv, scoring='accuracy', n_jobs=-1)

print(f"5-Fold CV Accuracy Scores: {[round(s * 100, 2) for s in cv_scores]} %")
print(f"Mean CV Accuracy: {round(cv_scores.mean() * 100, 2)}% (+/- {round(cv_scores.std() * 100, 2)}%)")

Champion Model: Random Forest
5-Fold CV Accuracy Scores: [np.float64(97.85), np.float64(98.0), np.float64(98.92), np.float64(98.0), np.float64(98.15)] %
Mean CV Accuracy: 98.18% (+/- 0.38%)


In [7]:
rf = models["Random Forest"]
importances = rf.feature_importances_
feat_importance_series = pd.Series(importances, index=feature_names).sort_values(ascending=False)

print("Top 15 Most Influential Agronomic Features:")
display(feat_importance_series.head(15).round(4))

Top 15 Most Influential Agronomic Features:


growth_duration_days        0.1428
K                           0.0966
humidity                    0.0897
N                           0.0844
rainfall                    0.0729
P                           0.0658
temperature                 0.0602
sunlight_hours              0.0433
water_requirement_Low       0.0351
water_requirement_Medium    0.0345
ph                          0.0331
category_Vegetable          0.0260
category_Fruit              0.0240
category_Oilseed            0.0225
category_Cereal             0.0219
dtype: float64

In [8]:
def predict_top_crops(farm_profile: dict, top_k: int = 3):
    input_df = pd.DataFrame([farm_profile])
    input_encoded = pd.get_dummies(input_df, columns=categorical_cols, drop_first=False)
    
    # Ensure all feature columns match training schema
    for col in feature_names:
        if col not in input_encoded.columns:
            input_encoded[col] = 0
    input_encoded = input_encoded[feature_names]
    
    # Scale input
    input_scaled = scaler.transform(input_encoded)
    
    # Predict probabilities
    probs = best_model.predict_proba(input_scaled)[0]
    top_k_indices = np.argsort(probs)[::-1][:top_k]
    
    recommendations = []
    for rank, idx in enumerate(top_k_indices, 1):
        crop = class_names[idx]
        conf = round(float(probs[idx]) * 100, 2)
        recommendations.append({
            "rank": rank,
            "crop": crop,
            "confidence": f"{conf}%"
        })
    return recommendations

# Test Scenario A: High-rainfall paddy parcel
paddy_parcel = {
    'N': 82, 'P': 46, 'K': 40, 'temperature': 24.5, 'humidity': 83.0, 'ph': 6.4, 'rainfall': 230.0,
    'soil_type': 'Clayey', 'category': 'Cereal', 'growth_duration_days': 120,
    'water_requirement': 'Very High', 'sunlight_hours': 7.0
}

# Test Scenario B: Arid sandy millet parcel
millet_parcel = {
    'N': 58, 'P': 28, 'K': 24, 'temperature': 32.0, 'humidity': 40.0, 'ph': 7.2, 'rainfall': 42.0,
    'soil_type': 'Sandy', 'category': 'Cereal', 'growth_duration_days': 85,
    'water_requirement': 'Low', 'sunlight_hours': 9.0
}

print("=== Scenario A (High Moisture) ===")
print("Predictions:", predict_top_crops(paddy_parcel))

print("\n=== Scenario B (Semi-Arid Millet) ===")
print("Predictions:", predict_top_crops(millet_parcel))

=== Scenario A (High Moisture) ===
Predictions: [{'rank': 1, 'crop': 'Rice', 'confidence': '100.0%'}, {'rank': 2, 'crop': 'Wheat', 'confidence': '0.0%'}, {'rank': 3, 'crop': 'Turmeric', 'confidence': '0.0%'}]

=== Scenario B (Semi-Arid Millet) ===
Predictions: [{'rank': 1, 'crop': 'Pearl Millet', 'confidence': '100.0%'}, {'rank': 2, 'crop': 'Wheat', 'confidence': '0.0%'}, {'rank': 3, 'crop': 'Turmeric', 'confidence': '0.0%'}]


In [9]:
export_dir = os.path.join("..", "ml_models")
if not os.path.exists(export_dir):
    export_dir = "ml_models"

artifacts = {
    "model": best_model,
    "scaler": scaler,
    "label_encoder": label_encoder,
    "feature_names": feature_names,
    "categorical_cols": categorical_cols,
    "numerical_cols": numerical_cols,
    "classes": class_names,
    "metadata": {
        "model_type": best_model_name,
        "num_classes": len(class_names),
        "total_features": len(feature_names),
        "test_accuracy": float(results_df.iloc[0]["Test Accuracy (%)"])
    }
}

artifact_path = os.path.join(export_dir, "best_crop_model.joblib")
joblib.dump(artifacts, artifact_path)
print(f"[SUCCESS] Production artifacts successfully serialized to: {os.path.abspath(artifact_path)}")
print(f"   Model Type   : {best_model_name}")
print(f"   Total Classes: {len(class_names)}")

[SUCCESS] Production artifacts successfully serialized to: d:\PROJECTS\Krishi-Sakhi\backend\ml_models\best_crop_model.joblib
   Model Type   : Random Forest
   Total Classes: 58
